# 1. Pytorch란?

- 딥러닝 모델을 쉽고 유연하게 만들 수 있도록 도와주는 파이썬 라이브러리
- NumPy와 매우 유사한 방식ㅇ로 동작
- 단, **GPU를 활용한 가속**과 **자동 미분(AutoGrad)**이라는 강력한 기능을 제공

## 1-1. 기본 텐서(Tensor)다루기

### 1-1-1. 텐서(Tensor)란?

- 다차원 배열을 의미.
    - 0차원 텐서 - 스칼라
    - 1차원 텐서 - 벡터
    - 2차원 텐서 - 행렬
    - 그 이상 - 고차원 텐서라고 부르는게 일반적
- NumPy의 `ndarray`와 매우 유사하지만, 딥러닝에 필수적인 2가지 추가 기능을 보유
    1. **GPU 가속:** 일반적인 방법인 CPU 연산이 아닌, GPU로 보내져 매우 빠른 병렬 연산 수행 가능
    2. **자동 미분:** 텐서는 자신이 거쳐온 연산을 기록하여, 미분값을 자동으로 계산 해 줌.

In [2]:
import torch

# NumPy와 유사한 텐서 생성 및 연산
x = torch.rand(3, 3)
y = torch.ones(3, 3)
print("x + y:\n", x + y)
print("x @ y.T:\n", x @ y.T)

x + y:
 tensor([[1.6931, 1.9959, 1.9468],
        [1.1311, 1.4246, 1.4551],
        [1.6230, 1.4988, 1.2781]])
x @ y.T:
 tensor([[2.6358, 2.6358, 2.6358],
        [1.0108, 1.0108, 1.0108],
        [1.3999, 1.3999, 1.3999]])


### 1-1-3. 자동 미분(AutoGrad)와 경사 하강법

- 모델이 학습하는 원리 중 하나인 `경사 하강법`를 위해, PyTorch가 어떻게 `기울기`를 계산하는지 알아봅시다.
1. **경사 하강법**과 **기울기(Gradiend)**
    - 모델 학습의 목표는 `손실 함수(Loss Funtion)의 값을 최소하` 하는 최적의 파라미터 (가중치 W와 편향 b)를 찾는 것
    - 이 손실을 최소화 하기 위해 가장 널리 사용되는 최적화 알고리즘이 경사 하강법
    - 손실 함수의 `기울기(Gradient)`를 이용해, 손실 값이 가장 가파르게 감소하는 방향으로 파라미터를 점진적으로 업데이트 하는 것
    - 경사 하강법: $\theta new = \theta old - \eta \nabla_\theta L$
        - **$\theta$(세타):** 모델의 파라미터 (가중치 W와 편향 b)
        - **$\eta$(에타):** 학습률(Learning rage). 기울기 방향으로 얼마나 크게 이동 시킬지 결정하는 값
        - $L$: 손실 함수 (Loss Function)
            - 이진 교차 엔트로피
                - 모델이 예측한 확률이 실제 레이블과 얼마나 일치하는지 측정
            - 범주형 교차 엔트로피
                - 모델이 출력한 확률 분포가 실제 레이블의 분포와 얼마나 가까운지 측정
        - **$\nabla_\theta L$:** 손실 함수 L을 파라미터로 편미분한 기울기
            - 손실이 가장 크게 `증가` 하는 방향을 나타내므로, 우리는 이 크기를 줄이기를 바라므로 이전 파라미터에 기울기 값 만큼을 제거 할 것.
                - 단, 가장 큰 잔차를 그대로 제거해 버리면 너무 큰 보폭이 되어버리므로
                적절한 $\eta$ 학습률을 설정할 필요가 있음.
            - 가중치와 편향이 각각 얼마나 영향을 미치는지를 보기위해 각각 편미분 하는 것
2. **AutoGrad (자동 경사도 계산)**
    - 즉, **$\nabla_\theta L$** 부분을 자동으로 계산해 준다는 것.
    - 텐서의 모든 연산과정을 그래프 형태로 기록 해 둠.
    - 이후 **역전파(backward)**가 진행될 때, 이 그래프를 따라 각 파라미터에 대한 기울기를 자동으로 계산해 줌.

3. 코드 예시
    

In [ ]:
# requires_grad=True로 설정하여 연산 과정을 추적
    # 어떤 정보를 저장하는가?
    # - 연산 그래프(Computational Graph)
        # 각 연산과 텐서의 관계를 나타내는 그래프
    # - 각 텐서의 기울기(Gradient, ∇)
    # - 역전파(Backpropagation)를 통해 기울기를 계산하는 방법  
a = torch.tensor([2.0, 3.0], requires_grad=True)
b = a ** 2 + 3 * a + 1
z = b.sum()

# z를 a로 미분 (역전파)
z.backward()

# 계산된 기울기(Gradient, ∇) 확인
print(a.grad) # .grad 속성에 기울기가 저장

tensor([7., 9.])


## 1-2. 응용: 선형 회귀 모델의 기울기 계산하기

- AutoGrad를 실제 머신러닝 모델에 적용하여 가중치와 편향에 대한 기울기 계산하기
1. MSE 손실 함수를 기준으로 각 파라미터가 오차에 얼마나 영향을 미치는가를 계산
    1. 영향력(기울기)를 `.backward()` 함수를 사용해 간단하게 계산하는 방법!
- 코드

In [4]:
# 학습할 파라미터 정의
weight = torch.tensor([[3.0]], requires_grad=True)
bias   = torch.tensor([[1.0]], requires_grad=True)

# 데이터
x = torch.tensor([[2.0]])
y_true = torch.tensor([[4.0]])

# 순전파 및 손실 계산
y_pred = x @ weight + bias
loss = torch.mean((y_pred - y_true) ** 2)

# 역전파 (자동 미분)
loss.backward()

# 각 파라미터에 대한 기울기 확인
print("Weight의 기울기:", weight.grad)
print("Bias의 기울기:", bias.grad)

Weight의 기울기: tensor([[12.]])
Bias의 기울기: tensor([[6.]])
